In [ ]:
# ================================
# 1️⃣ Install Dependencies
# ================================
!pip install transformers datasets scikit-learn tqdm

# ================================
# 2️⃣ Imports
# ================================
import json
from tqdm import tqdm
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# For auto-download
from google.colab import files

# ================================
# 3️⃣ Load Dataset
# ================================
file_path = "/content/ML_relations_fixed.json"

with open(file_path, "r") as f:
    data = json.load(f)

print("Total samples:", len(data))

# ================================
# 4️⃣ Labels (Auto from Data)
# ================================
LABELS = sorted(list(set([ex["output"].strip() for ex in data])))
print("Labels:", LABELS)

# ================================
# 5️⃣ Load Zero-Shot RoBERTa
# ================================
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",   # RoBERTa-style NLI model
    device=0  # use GPU if available, else remove this line
)

# ================================
# 6️⃣ Prediction Function
# ================================
def predict(text):
    result = classifier(
        text,
        candidate_labels=LABELS,
        hypothesis_template="This sentence expresses {}."
    )
    return result["labels"][0]   # top prediction

# ================================
# 7️⃣ Run Predictions
# ================================
y_true = []
y_pred = []
results = []

for ex in tqdm(data):
    text = ex["input"]
    gt = ex["output"].strip()

    pred = predict(text)

    y_true.append(gt)
    y_pred.append(pred)

    results.append({
        "input": text,
        "ground_truth": gt,
        "predicted": pred
    })

# ================================
# 8️⃣ Save Results
# ================================
output_file = "/content/roberta_zero_shot_results.json"

with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

files.download(output_file)

print(f"\n✅ Results saved to: {output_file}")

# ================================
# 9️⃣ Metrics
# ================================
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n===== ROBERTA ZERO-SHOT RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== Classification Report =====")
print(classification_report(y_true, y_pred))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_true, y_pred))

# ================================
# 🔟 Show Predictions
# ================================
for i in range(min(5, len(results))):
    print("\n--- Example ---")
    print("Input :", results[i]["input"])
    print("GT    :", results[i]["ground_truth"])
    print("Pred  :", results[i]["predicted"])

Total samples: 1962
Labels: ['based_on', 'implements', 'improves', 'no_relation', 'part_of', 'used_for']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

100%|██████████| 1962/1962 [04:43<00:00,  6.91it/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Results saved to: /content/roberta_zero_shot_results.json

===== ROBERTA ZERO-SHOT RESULTS =====
Accuracy : 0.1952
Precision: 0.4143
Recall   : 0.1952
F1 Score : 0.1282

===== Classification Report =====
              precision    recall  f1-score   support

    based_on       0.20      0.03      0.06        90
  implements       0.12      0.21      0.15       170
    improves       0.30      0.42      0.35       242
 no_relation       0.18      0.01      0.01       584
     part_of       0.17      0.75      0.27       285
    used_for       0.93      0.05      0.09       591

    accuracy                           0.20      1962
   macro avg       0.32      0.24      0.16      1962
weighted avg       0.41      0.20      0.13      1962


===== Confusion Matrix =====
[[  3  14   4   1  68   0]
 [  1  35  24   1 108   1]
 [  0  16 102   0 124   0]
 [ 11 123  93   3 353   1]
 [  0  27  45   0 213   0]
 [  0  81  67  12 404  27]]

--- Example ---
Input : Sentence: Reinforcement Learning